# 06. CHEESE Temporal EDA

## Mục tiêu

> Tái sử dụng các hàm EDA đã chạy trên toàn bộ transaction data để mô tả riêng hành vi mua `CHEESE`.

Notebook giữ nguyên định nghĩa lifecycle, recency, interpurchase interval, historical coverage, window comparison và behavior shift. Khác biệt duy nhất là đầu vào chỉ chứa product lines có `COMMODITY_DESC == "CHEESE"`. Vì vậy population trong các biểu đồ là household từng mua CHEESE, không phải toàn bộ 2.500 household.

In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

from pathlib import Path

import polars as pl

from propensity.eda import (
    BehaviorShiftSpec,
    build_household_behavior_shift,
    build_household_interpurchase_features,
    build_household_lifecycle,
    build_household_rolling_windows,
    profile_historical_activity_coverage,
    profile_return_intervals,
    profile_window_candidates_across_cutoffs,
)
from propensity.eda.dunnhumby import DUNNHUMBY_TRANSACTION_PROFILE_CONFIG
from propensity.eda.visualization import (
    plot_historical_activity_coverage,
    plot_household_behavior_change,
    plot_household_behavior_shift,
    plot_household_interpurchase_dynamics,
    plot_last_purchase_recency,
    plot_lifecycle_coverage_exposure,
    plot_return_interval_distribution,
    plot_window_candidate_evidence,
)
from propensity.io import scan_csv
from propensity.schema.dunnhumby import TRANSACTION_SCHEMA_OVERRIDES

pl.Config.set_tbl_rows(30)
pl.Config.set_tbl_cols(20)

## 1. Tạo transaction frame riêng cho CHEESE

Lọc product master trước rồi semi-join vào transaction data. Cách này giữ nguyên schema giao dịch mà các hàm EDA dùng, đồng thời không nhân bản product lines.

In [ ]:
project_root = Path.cwd().resolve()
if project_root.name == "notebooks":
    project_root = project_root.parent

data_dir = project_root / "data" / "raw" / "Dunnhumby"
artifact_dir = project_root / "output" / "06_cheese_temporal_eda"
artifact_dir.mkdir(parents=True, exist_ok=True)

config = DUNNHUMBY_TRANSACTION_PROFILE_CONFIG
transactions = scan_csv(
    data_dir / "transaction_data.csv",
    schema_overrides=TRANSACTION_SCHEMA_OVERRIDES,
)
cheese_products = (
    scan_csv(data_dir / "product.csv")
    .filter(pl.col("COMMODITY_DESC") == "CHEESE")
    .select("PRODUCT_ID")
    .unique()
)
cheese_transactions = transactions.join(
    cheese_products,
    on="PRODUCT_ID",
    how="semi",
)

In [ ]:
cheese_scope = cheese_transactions.select(
    pl.len().alias("n_product_lines"),
    pl.col(config.basket_col).n_unique().alias("n_baskets"),
    pl.col(config.entity_col).n_unique().alias("n_households"),
    pl.col("PRODUCT_ID").n_unique().alias("n_products"),
    pl.col(config.time_col).min().alias("min_day"),
    pl.col(config.time_col).max().alias("max_day"),
).collect()
cheese_scope

## 2. Tạo lại các evidence tables

Mọi bảng dưới đây dùng đúng helper của notebook 04. `H = 14` được dùng trong so sánh candidate windows để khớp thiết kế thời gian đang đề xuất cho bài toán repeat purchase.

In [ ]:
cheese_lifecycle = build_household_lifecycle(cheese_transactions, config)
cheese_interpurchase = build_household_interpurchase_features(
    cheese_transactions,
    config,
)
cheese_return_intervals = profile_return_intervals(cheese_transactions, config)
cheese_activity_coverage = profile_historical_activity_coverage(
    cheese_transactions,
    config,
    windows_days=(3, 7, 14, 21, 30, 60, 90, 180, 365),
)
cheese_window_candidates = profile_window_candidates_across_cutoffs(
    cheese_transactions,
    config,
    windows_days=(7, 14, 30, 60, 90, 180),
    target_horizon_days=14,
    cutoff_step_days=7,
)
cheese_behavior_shift = build_household_behavior_shift(
    cheese_transactions,
    config,
    BehaviorShiftSpec(
        recent_days=30, history_days=90, cutoff_step_days=7
    ),
)
cheese_rolling_windows = build_household_rolling_windows(
    cheese_transactions,
    config,
    window_weeks=13,
    step_weeks=4,
)

## 3. Entry cohort và exposure bias

Biểu đồ này kiểm tra người mua CHEESE xuất hiện từ sớm hay rải rác, và raw CHEESE basket count có tăng cơ học theo số ngày được quan sát hay không.

In [ ]:
lifecycle_figure = plot_lifecycle_coverage_exposure(cheese_lifecycle)
lifecycle_figure.savefig(
    artifact_dir / "cheese_lifecycle_coverage_exposure.png",
    dpi=150,
    bbox_inches="tight",
)
lifecycle_figure

## 4. Last purchase và recency của người mua CHEESE

Recency được tính tại snapshot cuối của CHEESE data. Các tỷ lệ 7/14/30/60/90 ngày vì vậy chỉ mô tả household từng mua CHEESE.

In [ ]:
recency_figure = plot_last_purchase_recency(cheese_lifecycle)
recency_figure.savefig(
    artifact_dir / "cheese_last_purchase_recency.png",
    dpi=150,
    bbox_inches="tight",
)
recency_figure

## 5. Khoảng quay lại và regularity

Panel đầu nhìn phân phối event-level giữa hai CHEESE purchase days liên tiếp. Panel sau để mỗi household đóng góp một cadence điển hình, đồng thời so cadence với độ biến động và mức độ trễ tại snapshot.

In [ ]:
return_interval_figure = plot_return_interval_distribution(
    cheese_return_intervals
)
return_interval_figure.savefig(
    artifact_dir / "cheese_return_interval_distribution.png",
    dpi=150,
    bbox_inches="tight",
)
return_interval_figure

In [ ]:
interpurchase_figure = plot_household_interpurchase_dynamics(
    cheese_interpurchase,
    cheese_return_intervals,
)
interpurchase_figure.savefig(
    artifact_dir / "cheese_household_interpurchase_dynamics.png",
    dpi=150,
    bbox_inches="tight",
)
interpurchase_figure

## 6. Historical activity coverage

So sánh lượng CHEESE evidence có trong từng lookback tại cùng cutoff. Household từng mua CHEESE nhưng không mua trong trailing window vẫn được giữ với basket count bằng zero.

In [ ]:
coverage_figure = plot_historical_activity_coverage(cheese_activity_coverage)
coverage_figure.savefig(
    artifact_dir / "cheese_historical_activity_coverage.png",
    dpi=150,
    bbox_inches="tight",
)
coverage_figure

## 7. Candidate lookback windows với H = 14

Các candidate `T ∈ {7, 14, 30, 60, 90, 180}` được so sánh trên cùng historical cutoff set. Biểu đồ đánh giá coverage, stability và số cutoff khả dụng trong riêng CHEESE buyer cohort.

In [ ]:
window_figure = plot_window_candidate_evidence(cheese_window_candidates)
window_figure.savefig(
    artifact_dir / "cheese_window_candidate_evidence.png",
    dpi=150,
    bbox_inches="tight",
)
window_figure

## 8. Recent behavior shift

Qua các cutoff cách nhau 7 ngày, so sánh 30 ngày gần nhất với 90 ngày lịch sử không chồng lấn trên cùng cohort. Các transition mô tả thay đổi tần suất mua CHEESE, chưa phải churn states hay causal outcomes.

In [ ]:
shift_figure = plot_household_behavior_shift(cheese_behavior_shift)
shift_figure.savefig(
    artifact_dir / "cheese_household_behavior_shift.png",
    dpi=150,
    bbox_inches="tight",
)
shift_figure

## 9. Thay đổi cadence theo rolling windows

Rolling window 13 tuần, cập nhật mỗi 4 tuần, cho biết cadence của CHEESE buyers có ổn định qua lịch sử hay không.

In [ ]:
trend_figure = plot_household_behavior_change(cheese_rolling_windows)
trend_figure.savefig(
    artifact_dir / "cheese_household_behavior_change.png",
    dpi=150,
    bbox_inches="tight",
)
trend_figure

## 10. Cách đọc kết quả

Notebook này trả lời liệu các kết luận temporal ở cấp toàn bộ giao dịch có còn đúng trong CHEESE buyer cohort hay không. Khi diễn giải, cần so ba nhóm evidence:

- `H = 14` có bao phủ hợp lý purchase cadence riêng của CHEESE hay không.
- `T = 90` có đủ active purchase days và baskets để tạo cadence features hay không.
- Các kết luận có ổn định qua common historical cutoffs và rolling windows hay không.

Các biểu đồ chưa đo target lift, model performance hoặc tác động causal của promotion.